# YZM304 Derin Öğrenme – I. Proje Ödevi
## BankNote Authentication Veri Seti ile Binary Sınıflandırma
**Ankara Üniversitesi | Yapay Zeka ve Veri Mühendisliği | 2025-2026 Bahar**

> **Not:** Bu çalışma, 13.03.2026 tarihinde laboratuvar saatlerinde başlatılan
> `One_Hidden_Layer_MLP` modelinin devamıdır. Lab'da kullanılan veri seti, ağırlık başlatma yöntemi,
> hiperparametreler ve SGD optimizer aynen korunmuştur.

## 1. Kütüphaneler ve Yapılandırma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# ── Lab ile aynı hiperparametreler ──────────────────────────────────────────
SEED      = 42          # np.random.seed(42)  – lab ile aynı
LR        = 0.01        # learning_rate = 0.01 – lab ile aynı
N_H_LAB   = 6           # lab'da kullanılan gizli nöron sayısı
N_STEPS   = 500         # lab'da 500 adım
THRESHOLD = 0.90        # model seçim eşiği (%90 accuracy)

np.random.seed(SEED)
torch.manual_seed(SEED)
print("Tüm kütüphaneler yüklendi.")


## 2. Veri Yükleme ve Keşifsel Veri Analizi (EDA)

> **Veri Seti:** BankNote Authentication – 1372 örnek, 4 özellik.
> Banknot görüntülerinden türetilen dalgacık dönüşümü özellikleri ile
> gerçek / sahte banknotları ayırt etmek için kullanılır. (Lab veri seti)

In [ ]:
# Lab ile aynı veri yükleme adımı
df = pd.read_csv("BankNote_Authentication.csv")
df.info()


In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Sınıf dağılımı
print("Sınıf dağılımı:")
print(df.iloc[:, -1].value_counts())
print(f"\nSınıf dengesi: {df.iloc[:,-1].mean():.3f} (0=Sahte, 1=Gerçek)")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
feature_names = df.columns[:-1].tolist()
target        = df.columns[-1]

# 1) Sınıf dağılımı
ax = axes[0, 0]
counts = df[target].value_counts().sort_index()
ax.bar(['Sahte (0)', 'Gerçek (1)'], counts.values,
       color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax.set_title('Sınıf Dağılımı', fontsize=13)
ax.set_ylabel('Örnek Sayısı')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

# 2) Özellik dağılımları (histogram)
ax = axes[0, 1]
for feat in feature_names:
    ax.hist(df[feat], bins=30, alpha=0.5, label=feat)
ax.set_title('Özellik Dağılımları')
ax.legend(fontsize=8)
ax.set_xlabel('Değer')

# 3) Korelasyon ısı haritası
ax = axes[1, 0]
corr = df.corr()
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            linewidths=0.5, annot=True, fmt='.2f', fontsize=8)
ax.set_title('Korelasyon Matrisi')

# 4) Boxplot özellik vs sınıf
ax = axes[1, 1]
df_melt = df.melt(id_vars=target, value_vars=feature_names,
                  var_name='Özellik', value_name='Değer')
sns.boxplot(data=df_melt, x='Özellik', y='Değer', hue=target,
            palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_title('Sınıfa Göre Özellik Dağılımı')
ax.tick_params(axis='x', rotation=15)

plt.suptitle('Keşifsel Veri Analizi – BankNote Authentication',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print("eda_plots.png kaydedildi.")


## 3. Veri Ön İşleme

In [ ]:
# Lab ile aynı: shuffle → split
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df.iloc[:, :-1].to_numpy()   # (1372, 4)
y = df.iloc[:,  -1].to_numpy().reshape(-1, 1)  # (1372, 1)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


In [ ]:
# Eğitim / Doğrulama / Test bölme (70 / 15 / 15) – stratified
X_tmp,   X_test,  y_tmp,   y_test  = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_dev,   y_train, y_dev   = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=SEED, stratify=y_tmp)
# 0.1765 * 0.85 ≈ 0.15

print(f"Train : {X_train.shape}  |  Dev : {X_dev.shape}  |  Test : {X_test.shape}")

for split_name, split_y in [("Train", y_train), ("Dev", y_dev), ("Test", y_test)]:
    unique, counts = np.unique(split_y, return_counts=True)
    print(f"{split_name} sınıf dağılımı: {dict(zip(unique.astype(int), counts))}")


> **Not:** Lab'da `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`
> kullanılmıştır. Ödevde ek olarak **dev (doğrulama) seti** oluşturulmuştur.
> Regularizasyon ve overfitting analizleri için gereklidir.

## 4. NumPy ile Sinir Ağı (Lab Kodu Temelli)

### 4.1 Lab Fonksiyonları (13.03.2026)

Aşağıdaki `initialize_parameters`, `sigmoid`, `forward_propagation`,
`compute_cost`, `backpropagation`, `update_parameters`, `nn_model` ve `predict`
fonksiyonları lab'da verilen iskelet kodun tamamlanmış halidir.
Ağırlıklar **`np.random.seed(42)`** ve **`* 0.01`** ile başlatılmış,
optimizer olarak **SGD** (`learning_rate=0.01`) kullanılmıştır.

In [ ]:
# ── Lab ile birebir aynı: initialize_parameters ──────────────────────────
def initialize_parameters(n_x, n_h, n_y=1):
    np.random.seed(42)                          # Lab ile aynı seed
    W1 = np.random.randn(n_h, n_x) * 0.01      # (n_h, n_x)
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01      # (n_y, n_h)
    b2 = np.zeros((n_y, 1))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

# ── Sigmoid aktivasyon ────────────────────────────────────────────────────
def sigmoid(Z):
    return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

# ── İleri yayılım ────────────────────────────────────────────────────────
def forward_propagation(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    # X: (m, n_x)  →  W1: (n_h, n_x)
    Z1 = W1 @ X.T + b1      # (n_h, m)
    A1 = sigmoid(Z1)         # (n_h, m)
    Z2 = W2 @ A1 + b2        # (n_y, m)
    A2 = sigmoid(Z2)         # (n_y, m)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# ── Binary Cross-Entropy maliyet ─────────────────────────────────────────
def compute_cost(A2, Y):
    # A2: (1, m), Y: (m, 1)
    m    = A2.shape[1]
    A2c  = np.clip(A2, 1e-9, 1 - 1e-9)
    cost = -(np.dot(np.log(A2c), Y) + np.dot(np.log(1 - A2c), (1 - Y))) / m
    return float(np.squeeze(cost))

# ── Geri yayılım ──────────────────────────────────────────────────────────
def backpropagation(X, Y, cache, parameters):
    m             = X.shape[0]
    W2            = parameters["W2"]
    A1, A2        = cache["A1"], cache["A2"]
    # Y: (m,1)  →  dZ2: (1, m)
    dZ2 = A2 - Y.T                        # (1, m)
    dW2 = dZ2 @ A1.T / m                  # (1, n_h)
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m  # (1, 1)
    dA1 = W2.T @ dZ2                      # (n_h, m)
    dZ1 = dA1 * A1 * (1 - A1)            # sigmoid türevi: (n_h, m)
    dW1 = dZ1 @ X / m                     # (n_h, n_x)
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m  # (n_h, 1)
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

# ── SGD parametre güncelleme ──────────────────────────────────────────────
def update_parameters(parameters, grads, learning_rate=0.01):
    parameters["W1"] -= learning_rate * grads["dW1"]
    parameters["b1"] -= learning_rate * grads["db1"]
    parameters["W2"] -= learning_rate * grads["dW2"]
    parameters["b2"] -= learning_rate * grads["db2"]
    return parameters

# ── Tüm adımların entegrasyonu: nn_model ─────────────────────────────────
def nn_model(X, Y, n_x, n_h, n_y=1, n_steps=500, learning_rate=0.01,
             print_cost=True, X_dev=None, Y_dev=None):
    parameters = initialize_parameters(n_x, n_h, n_y)
    history    = {"train_loss": [], "dev_loss": [], "train_acc": [], "dev_acc": []}
    steps_to_90 = None

    for i in range(1, n_steps + 1):
        A2, cache  = forward_propagation(X, parameters)
        cost       = compute_cost(A2, Y)
        grads      = backpropagation(X, Y, cache, parameters)
        parameters = update_parameters(parameters, grads, learning_rate)

        tr_acc = float(np.mean((A2 >= 0.5).T == Y))
        history["train_loss"].append(cost)
        history["train_acc"].append(tr_acc)

        if X_dev is not None:
            A2_dev, _ = forward_propagation(X_dev, parameters)
            dev_loss  = compute_cost(A2_dev, Y_dev)
            dev_acc   = float(np.mean((A2_dev >= 0.5).T == Y_dev))
            history["dev_loss"].append(dev_loss)
            history["dev_acc"].append(dev_acc)
            if steps_to_90 is None and dev_acc >= THRESHOLD:
                steps_to_90 = i

        if print_cost and i % 100 == 0:
            msg = f"  Adım {i:4d} | Train Loss: {cost:.4f} | Train Acc: {tr_acc:.4f}"
            if X_dev is not None:
                msg += f" | Dev Loss: {dev_loss:.4f} | Dev Acc: {dev_acc:.4f}"
            print(msg)

    return parameters, history, steps_to_90

# ── Tahmin ────────────────────────────────────────────────────────────────
def predict(parameters, X):
    A2, _ = forward_propagation(X, parameters)
    return (A2 >= 0.5).astype(int)  # (1, m)

print("Lab fonksiyonları tanımlandı.")


### 4.2 Lab Modeli Eğitimi (1 Gizli Katman, n_h=6, n_steps=500)

Lab'da kullanılan hiperparametrelerle (`n_h=6`, `lr=0.01`, `n_steps=500`) model eğitiliyor.

In [ ]:
print("=" * 60)
print(f"  Lab Modeli | Mimari: [{X_train.shape[1]}, {N_H_LAB}, 1]")
print(f"  lr={LR} | n_steps={N_STEPS} | seed={SEED} | init: *0.01")
print("=" * 60)

lab_params, lab_history, lab_steps90 = nn_model(
    X_train, y_train,
    n_x=X_train.shape[1], n_h=N_H_LAB, n_y=1,
    n_steps=N_STEPS, learning_rate=LR,
    print_cost=True, X_dev=X_dev, Y_dev=y_dev
)
print(f"\n  ► 90% dev acc eşiğine ulaşılan adım: {lab_steps90 if lab_steps90 else 'N/A'}")


### 4.3 Overfitting / Underfitting Analizi

In [ ]:
def evaluate(parameters, X, y):
    preds = predict(parameters, X).T      # (m, 1)
    y_flat = y.flatten()
    p_flat = preds.flatten()
    return {
        "accuracy" : accuracy_score(y_flat, p_flat),
        "precision": precision_score(y_flat, p_flat, zero_division=0),
        "recall"   : recall_score(y_flat, p_flat, zero_division=0),
        "f1"       : f1_score(y_flat, p_flat, zero_division=0),
    }

tr  = evaluate(lab_params, X_train, y_train)
dv  = evaluate(lab_params, X_dev,   y_dev)
tst = evaluate(lab_params, X_test,  y_test)

gap = tr['accuracy'] - dv['accuracy']
if   gap > 0.05:              durum = "Overfitting (Yüksek Varyans)"
elif tr['accuracy'] < 0.85:   durum = "Underfitting (Yüksek Bias)"
else:                          durum = "İyi Fit"

print(f"{'Set':<10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 50)
for name, m in [("Train", tr), ("Dev", dv), ("Test", tst)]:
    print(f"{name:<10} {m['accuracy']:>10.4f} {m['precision']:>10.4f} "
          f"{m['recall']:>10.4f} {m['f1']:>10.4f}")
print(f"\nTrain-Dev Acc Farkı: {gap:.4f}  →  {durum}")


### 4.4 Öğrenme Eğrileri – Lab Modeli

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
steps = range(1, len(lab_history['train_loss']) + 1)

ax = axes[0]
ax.plot(steps, lab_history['train_loss'], color='#3498db', label='Train Loss', lw=1.5)
ax.plot(steps, lab_history['dev_loss'],   color='#e74c3c', label='Dev Loss',   lw=1.5, ls='--')
ax.set_title('Loss Eğrisi – Lab Modeli (1 Gizli Katman)')
ax.set_xlabel('Adım'); ax.set_ylabel('BCE Loss'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(steps, lab_history['train_acc'], color='#3498db', label='Train Acc', lw=1.5)
ax.plot(steps, lab_history['dev_acc'],   color='#e74c3c', label='Dev Acc',   lw=1.5, ls='--')
ax.axhline(THRESHOLD, color='gray', lw=1.2, ls=':', label=f'{THRESHOLD*100:.0f}% eşiği')
ax.set_title('Accuracy Eğrisi – Lab Modeli (1 Gizli Katman)')
ax.set_xlabel('Adım'); ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Öğrenme Eğrileri – Lab Modeli', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("learning_curves.png kaydedildi.")


### 4.5 Model Seçimi: Grid Search (Lab'dan)

Lab'daki son blok: `n_h` ve `n_steps` üzerinde grid search. Kriter: **%90 accuracy'yi geçen modellerde `n_steps` en küçük olan.**

In [ ]:
parameters_n_h     = list(range(3, 11))
parameters_n_steps = list(range(100, 1100, 100))

grid_results = []
print(f"{'n_h':>5} {'n_steps':>8} {'Train Acc':>10} {'Dev Acc':>10} {'Test Acc':>10}")
print("-" * 48)

for n_h in parameters_n_h:
    for n_step in parameters_n_steps:
        params, _, _ = nn_model(
            X_train, y_train,
            n_x=X_train.shape[1], n_h=n_h, n_y=1,
            n_steps=n_step, learning_rate=LR,
            print_cost=False
        )
        tr_acc  = evaluate(params, X_train, y_train)['accuracy']
        tst_acc = evaluate(params, X_test,  y_test)['accuracy']
        dv_acc  = evaluate(params, X_dev,   y_dev)['accuracy']
        grid_results.append({
            "n_h": n_h, "n_steps": n_step,
            "train_acc": tr_acc, "dev_acc": dv_acc, "test_acc": tst_acc,
            "params": params
        })
        print(f"{n_h:>5} {n_step:>8} {tr_acc:>10.4f} {dv_acc:>10.4f} {tst_acc:>10.4f}")


In [ ]:
# Kriter: dev_acc >= 0.90 olanlar arasında n_steps en küçük
candidates = [r for r in grid_results if r['dev_acc'] >= THRESHOLD]

if candidates:
    best = min(candidates, key=lambda r: r['n_steps'])
else:
    best = max(grid_results, key=lambda r: r['dev_acc'])

print(f"\nSeçilen Model:")
print(f"  n_h     : {best['n_h']}")
print(f"  n_steps : {best['n_steps']}")
print(f"  Train   : {best['train_acc']:.4f}")
print(f"  Dev     : {best['dev_acc']:.4f}")
print(f"  Test    : {best['test_acc']:.4f}")
best_params_1h = best['params']
best_n_h       = best['n_h']
best_n_steps   = best['n_steps']


### 4.6 Daha Derin Modeller (Ödev Gereksinimi)

Ödev: *'gizli katman sayısı 1 artırılmalı ya da nöron sayıları artırılmalıdır.'*
Aşağıda 2 gizli katmanlı model ekleniyor.

In [ ]:
class NeuralNetwork:
    """
    NumPy ile yazılmış çok katmanlı MLP.
    Lab'ın sigmoid aktivasyonunu ve SGD optimizer'ını kullanır.
    L2 regularizasyon (weight_decay) eklenmiştir.
    """

    def __init__(self, layer_sizes, lr=0.01, n_steps=500, seed=42, weight_decay=0.0):
        self.layer_sizes  = layer_sizes
        self.lr           = lr
        self.n_steps      = n_steps
        self.seed         = seed
        self.weight_decay = weight_decay
        self.params       = {}
        self.history      = {"train_loss": [], "dev_loss": [],
                             "train_acc":  [], "dev_acc":  []}
        self.steps_to_90  = None
        self._init_weights()

    # ── Private ──────────────────────────────────────────────────────────
    def _init_weights(self):
        """Lab ile uyumlu: seed=42, *0.01 başlatma."""
        np.random.seed(self.seed)
        for i in range(1, len(self.layer_sizes)):
            n_prev = self.layer_sizes[i - 1]
            n_curr = self.layer_sizes[i]
            self.params[f'W{i}'] = np.random.randn(n_curr, n_prev) * 0.01
            self.params[f'b{i}'] = np.zeros((n_curr, 1))

    @staticmethod
    def _sigmoid(Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    def _forward(self, X):
        cache = {'A0': X.T}  # (n_x, m)
        n_layers = len(self.layer_sizes) - 1
        for i in range(1, n_layers + 1):
            Z = self.params[f'W{i}'] @ cache[f'A{i-1}'] + self.params[f'b{i}']
            A = self._sigmoid(Z)
            cache[f'Z{i}'] = Z
            cache[f'A{i}'] = A
        return cache

    def _loss(self, A_out, Y):
        m   = Y.shape[0]
        Ac  = np.clip(A_out, 1e-9, 1 - 1e-9)
        bce = -np.mean(Y.T * np.log(Ac) + (1 - Y.T) * np.log(1 - Ac))
        l2  = 0.0
        if self.weight_decay > 0:
            for k, v in self.params.items():
                if k.startswith('W'):
                    l2 += np.sum(v ** 2)
            l2 *= self.weight_decay / (2 * m)
        return float(bce + l2)

    def _backward(self, X, Y, cache):
        m        = X.shape[0]
        n_layers = len(self.layer_sizes) - 1
        grads    = {}
        A_out    = cache[f'A{n_layers}']
        # dZ for sigmoid output: A - Y
        dA = A_out - Y.T          # (n_y, m)
        for i in range(n_layers, 0, -1):
            A_prev = cache[f'A{i-1}']
            dW = dA @ A_prev.T / m
            if self.weight_decay > 0:
                dW += (self.weight_decay / m) * self.params[f'W{i}']
            db = np.sum(dA, axis=1, keepdims=True) / m
            grads[f'W{i}'] = dW
            grads[f'b{i}'] = db
            if i > 1:
                A_i = cache[f'A{i-1}']
                dA  = (self.params[f'W{i}'].T @ dA) * A_i * (1 - A_i)
        return grads

    def _update(self, grads):
        for k in grads:
            self.params[k] -= self.lr * grads[k]

    # ── Public ───────────────────────────────────────────────────────────
    def fit(self, X_train, y_train, X_dev=None, y_dev=None, verbose=100):
        for step in range(1, self.n_steps + 1):
            cache     = self._forward(X_train)
            n_l       = len(self.layer_sizes) - 1
            A_out     = cache[f'A{n_l}']
            loss      = self._loss(A_out, y_train)
            tr_acc    = float(np.mean((A_out >= 0.5).T == y_train))
            grads     = self._backward(X_train, y_train, cache)
            self._update(grads)
            self.history['train_loss'].append(loss)
            self.history['train_acc'].append(tr_acc)
            if X_dev is not None:
                dc      = self._forward(X_dev)
                d_out   = dc[f'A{n_l}']
                d_loss  = self._loss(d_out, y_dev)
                d_acc   = float(np.mean((d_out >= 0.5).T == y_dev))
                self.history['dev_loss'].append(d_loss)
                self.history['dev_acc'].append(d_acc)
                if self.steps_to_90 is None and d_acc >= THRESHOLD:
                    self.steps_to_90 = step
            if step % verbose == 0:
                msg = f"  Adım {step:4d} | Loss: {loss:.4f} | Train: {tr_acc:.4f}"
                if X_dev is not None:
                    msg += f" | Dev: {d_acc:.4f}"
                print(msg)
        return self

    def predict(self, X):
        cache = self._forward(X)
        n_l   = len(self.layer_sizes) - 1
        return (cache[f'A{n_l}'] >= 0.5).astype(int).T  # (m, 1)

    def evaluate(self, X, y):
        p = self.predict(X).flatten()
        g = y.flatten()
        return {
            "accuracy" : accuracy_score(g, p),
            "precision": precision_score(g, p, zero_division=0),
            "recall"   : recall_score(g, p, zero_division=0),
            "f1"       : f1_score(g, p, zero_division=0),
        }

print("NeuralNetwork sınıfı tanımlandı.")


In [ ]:
n_x = X_train.shape[1]  # 4

architectures = {
    "M1_Lab_1H"  : [n_x, N_H_LAB, 1],           # Lab modeli
    "M2_2H"      : [n_x, 16, 8, 1],              # 2 gizli katman
    "M3_2H_Large": [n_x, 32, 16, 1],             # 2 gizli daha büyük
}

trained_models = {}
for name, arch in architectures.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}  |  Mimari: {arch}")
    print('='*60)
    model = NeuralNetwork(layer_sizes=arch, lr=LR, n_steps=N_STEPS,
                          seed=SEED, weight_decay=0.0)
    model.fit(X_train, y_train, X_dev, y_dev, verbose=100)
    trained_models[name] = model
    s90 = model.steps_to_90
    print(f"  ► 90% dev acc eşiği: {s90 if s90 else 'N/A'}")


In [ ]:
print(f"{'Model':<15} {'Train':>8} {'Dev':>8} {'Test':>8} {'Durum'}")
print("-" * 60)
for name, model in trained_models.items():
    tr  = model.evaluate(X_train, y_train)
    dv  = model.evaluate(X_dev,   y_dev)
    tst = model.evaluate(X_test,  y_test)
    gap = tr['accuracy'] - dv['accuracy']
    if   gap > 0.05:            durum = "Overfitting"
    elif tr['accuracy'] < 0.85: durum = "Underfitting"
    else:                        durum = "İyi Fit"
    print(f"{name:<15} {tr['accuracy']:>8.4f} {dv['accuracy']:>8.4f} {tst['accuracy']:>8.4f}  {durum}")


### 4.7 Öğrenme Eğrileri – Tüm Modeller

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
colors = {'M1_Lab_1H': '#e74c3c', 'M2_2H': '#2ecc71', 'M3_2H_Large': '#3498db'}

for col, (name, model) in enumerate(trained_models.items()):
    steps = range(1, len(model.history['train_loss']) + 1)
    ax = axes[0, col]
    ax.plot(steps, model.history['train_loss'], color=colors[name], label='Train', lw=1.5)
    ax.plot(steps, model.history['dev_loss'],   color=colors[name], ls='--', label='Dev', lw=1.5)
    ax.set_title(f'{name} – Loss'); ax.set_xlabel('Adım'); ax.set_ylabel('BCE Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1, col]
    ax.plot(steps, model.history['train_acc'], color=colors[name], label='Train', lw=1.5)
    ax.plot(steps, model.history['dev_acc'],   color=colors[name], ls='--', label='Dev', lw=1.5)
    ax.axhline(THRESHOLD, color='gray', lw=1, ls=':')
    ax.set_title(f'{name} – Accuracy'); ax.set_xlabel('Adım'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Öğrenme Eğrileri – Tüm Modeller', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("learning_curves.png kaydedildi.")


### 4.8 Model Seçimi

Kriter: dev_acc ≥ %90 olanlar arasında `steps_to_90` en küçük olan.

In [ ]:
candidates = {n: m for n, m in trained_models.items()
              if m.steps_to_90 is not None and
                 m.evaluate(X_dev, y_dev)['accuracy'] >= THRESHOLD}

if candidates:
    best_name = min(candidates, key=lambda n: candidates[n].steps_to_90)
else:
    best_name = max(trained_models, key=lambda n: trained_models[n].evaluate(X_dev, y_dev)['accuracy'])

best_model = trained_models[best_name]
print(f"Seçilen Model : {best_name}")
print(f"Mimari        : {best_model.layer_sizes}")
print(f"Steps-to-90%  : {best_model.steps_to_90}")
print(f"Dev Accuracy  : {best_model.evaluate(X_dev, y_dev)['accuracy']:.4f}")


### 4.9 Regülarizasyon Deneyi: L2 Weight Decay

In [ ]:
WD_VALUES = [0.0, 0.001, 0.01, 0.1]
reg_results = {}

print(f"{'L2 (λ)':<12} {'Train':>8} {'Dev':>8} {'Test':>8} {'Durum'}")
print("-" * 60)

for wd in WD_VALUES:
    m = NeuralNetwork(layer_sizes=best_model.layer_sizes, lr=LR,
                      n_steps=N_STEPS, seed=SEED, weight_decay=wd)
    m.fit(X_train, y_train, X_dev, y_dev, verbose=9999)
    tr  = m.evaluate(X_train, y_train)['accuracy']
    dv  = m.evaluate(X_dev,   y_dev)['accuracy']
    tst = m.evaluate(X_test,  y_test)['accuracy']
    gap = tr - dv
    if   gap > 0.05:  durum = "High Variance (Overfit)"
    elif tr  < 0.85:  durum = "High Bias (Underfit)"
    else:             durum = "İyi Fit"
    reg_results[wd] = {'train': tr, 'dev': dv, 'test': tst, 'durum': durum}
    print(f"λ={wd:<8} {tr:>8.4f} {dv:>8.4f} {tst:>8.4f}  {durum}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
wds    = list(reg_results.keys())
labels = [f'λ={w}' for w in wds]
trs    = [reg_results[w]['train'] for w in wds]
devs   = [reg_results[w]['dev']   for w in wds]
tsts   = [reg_results[w]['test']  for w in wds]

ax = axes[0]
x  = np.arange(len(wds)); bw = 0.25
ax.bar(x - bw, trs,  bw, label='Train', color='#3498db', alpha=0.85, edgecolor='black')
ax.bar(x,      devs, bw, label='Dev',   color='#e74c3c', alpha=0.85, edgecolor='black')
ax.bar(x + bw, tsts, bw, label='Test',  color='#2ecc71', alpha=0.85, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.8, 1.05); ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('L2 Regülarizasyon Etkisi', fontsize=12); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
gaps = [reg_results[w]['train'] - reg_results[w]['dev'] for w in wds]
cols = ['#e74c3c' if g > 0.05 else '#2ecc71' for g in gaps]
ax2.bar(labels, gaps, color=cols, edgecolor='black', alpha=0.85)
ax2.axhline(0.05, color='red', lw=1.2, ls='--', label='Overfitting eşiği (5%)')
ax2.set_ylabel('Train - Dev Acc (Varyans)'); ax2.legend(); ax2.grid(axis='y', alpha=0.3)
ax2.set_title('Varyans Analizi', fontsize=12)
for xi, g in enumerate(gaps):
    ax2.text(xi, g + 0.001, f'{g:.4f}', ha='center', fontsize=9)

plt.suptitle('Regülarizasyon Etkisi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('regularization_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("regularization_analysis.png kaydedildi.")


## 5. Scikit-learn MLPClassifier

In [ ]:
hidden_layers = tuple(best_model.layer_sizes[1:-1])

clf = MLPClassifier(
    hidden_layer_sizes=hidden_layers,
    activation='logistic',        # sigmoid – lab ile uyumlu
    solver='sgd',                 # SGD – lab ile uyumlu
    learning_rate_init=LR,
    max_iter=N_STEPS,
    random_state=SEED,
    alpha=0.001,                  # L2 regularizasyon
    batch_size=32,                # Mini-batch SGD
    early_stopping=False,
    n_iter_no_change=N_STEPS,
)
clf.fit(X_train, y_train.ravel())

sk_train = {
    "accuracy" : accuracy_score(y_train, clf.predict(X_train)),
    "precision": precision_score(y_train, clf.predict(X_train), zero_division=0),
    "recall"   : recall_score(y_train, clf.predict(X_train), zero_division=0),
    "f1"       : f1_score(y_train, clf.predict(X_train), zero_division=0),
}
sk_test = {
    "accuracy" : accuracy_score(y_test, clf.predict(X_test)),
    "precision": precision_score(y_test, clf.predict(X_test), zero_division=0),
    "recall"   : recall_score(y_test, clf.predict(X_test), zero_division=0),
    "f1"       : f1_score(y_test, clf.predict(X_test), zero_division=0),
}
print(f"Scikit-learn MLP | hidden={hidden_layers} | activation=logistic | solver=sgd")
print(f"Train Acc : {sk_train['accuracy']:.4f}")
print(f"Test  Acc : {sk_test['accuracy']:.4f}")


## 6. PyTorch ile Sinir Ağı

In [ ]:
class PyTorchNN(nn.Module):
    """PyTorch MLP – lab ile uyumlu: sigmoid aktivasyon, SGD."""

    def __init__(self, layer_sizes):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
            layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

pt_model  = PyTorchNN(best_model.layer_sizes)
criterion = nn.BCELoss()
pt_optim  = optim.SGD(pt_model.parameters(), lr=LR, weight_decay=1e-3)

X_tr_t  = torch.tensor(X_train, dtype=torch.float32)
y_tr_t  = torch.tensor(y_train, dtype=torch.float32)
X_tst_t = torch.tensor(X_test,  dtype=torch.float32)
y_tst_t = torch.tensor(y_test,  dtype=torch.float32)
X_dev_t = torch.tensor(X_dev,   dtype=torch.float32)
y_dev_t = torch.tensor(y_dev,   dtype=torch.float32)

BATCH_SIZE    = 32
train_dataset = TensorDataset(X_tr_t, y_tr_t)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           generator=torch.Generator().manual_seed(SEED))

print(f"PyTorch eğitimi: batch_size={BATCH_SIZE}, {len(train_loader)} batch/epoch")

pt_history = {"train_loss": [], "dev_loss": [], "train_acc": [], "dev_acc": []}
torch.manual_seed(SEED)

for epoch in range(1, N_STEPS + 1):
    pt_model.train()
    epoch_loss = 0.0
    for Xb, yb in train_loader:
        pt_optim.zero_grad()
        out  = pt_model(Xb)
        loss = criterion(out, yb)
        loss.backward()
        pt_optim.step()
        epoch_loss += loss.item() * Xb.size(0)
    epoch_loss /= len(X_tr_t)

    with torch.no_grad():
        pt_model.eval()
        tr_pred  = (pt_model(X_tr_t) >= 0.5).float()
        tr_acc   = (tr_pred == y_tr_t).float().mean().item()
        dev_out  = pt_model(X_dev_t)
        dev_loss = criterion(dev_out, y_dev_t).item()
        dev_pred = (dev_out >= 0.5).float()
        dev_acc  = (dev_pred == y_dev_t).float().mean().item()
        pt_history['train_loss'].append(epoch_loss)
        pt_history['dev_loss'].append(dev_loss)
        pt_history['train_acc'].append(tr_acc)
        pt_history['dev_acc'].append(dev_acc)
    if epoch % 100 == 0:
        print(f"  Epoch {epoch:4d} | Loss: {epoch_loss:.4f} | Train: {tr_acc:.4f} | Dev: {dev_acc:.4f}")


In [ ]:
with torch.no_grad():
    pt_preds = (pt_model(X_tst_t) >= 0.5).numpy().astype(int)

pt_test = {
    "accuracy" : accuracy_score(y_test, pt_preds),
    "precision": precision_score(y_test, pt_preds, zero_division=0),
    "recall"   : recall_score(y_test, pt_preds, zero_division=0),
    "f1"       : f1_score(y_test, pt_preds, zero_division=0),
}
print(f"PyTorch Test Accuracy : {pt_test['accuracy']:.4f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
steps_pt = range(1, N_STEPS + 1)
ax1.plot(steps_pt, pt_history['train_loss'], color='#e67e22', label='Train', lw=1.5)
ax1.plot(steps_pt, pt_history['dev_loss'],   color='#9b59b6', ls='--', label='Dev', lw=1.5)
ax1.set_title('PyTorch Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(steps_pt, pt_history['train_acc'], color='#e67e22', label='Train', lw=1.5)
ax2.plot(steps_pt, pt_history['dev_acc'],   color='#9b59b6', ls='--', label='Dev', lw=1.5)
ax2.axhline(THRESHOLD, color='gray', lw=1, ls=':')
ax2.set_title('PyTorch Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('PyTorch Öğrenme Eğrileri', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pytorch_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("pytorch_curves.png kaydedildi.")


## 7. Kapsamlı Değerlendirme

In [ ]:
numpy_preds = best_model.predict(X_test)
sk_preds    = clf.predict(X_test).reshape(-1, 1)

cms = {
    "NumPy NN"   : confusion_matrix(y_test, numpy_preds),
    "Scikit-learn": confusion_matrix(y_test, sk_preds),
    "PyTorch"    : confusion_matrix(y_test, pt_preds),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Sahte', 'Gerçek'],
                yticklabels=['Sahte', 'Gerçek'])
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Gerçek'); ax.set_xlabel('Tahmin')
plt.suptitle('Karmaşıklık Matrisleri – Test Seti', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print("confusion_matrices.png kaydedildi.")


In [ ]:
numpy_test = best_model.evaluate(X_test, y_test)
results_df = pd.DataFrame({
    "NumPy NN"   : numpy_test,
    "Scikit-learn": sk_test,
    "PyTorch"    : pt_test,
}).T.round(4)

print("\n── Test Seti Metrikleri ──")
print(results_df.to_string())
results_df


In [ ]:
metrics_plot = ['accuracy', 'precision', 'recall', 'f1']
models_plot  = ['NumPy NN', 'Scikit-learn', 'PyTorch']
vals = {m: [results_df.loc[m, k] for k in metrics_plot] for m in models_plot}

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_plot)); w = 0.25
cs = ['#3498db', '#e74c3c', '#2ecc71']
for i, (m, c) in enumerate(zip(models_plot, cs)):
    bars = ax.bar(x + i*w - w, vals[m], w, label=m, color=c, alpha=0.85, edgecolor='black')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1'])
ax.set_ylim(0.5, 1.1); ax.set_ylabel('Skor'); ax.legend()
ax.set_title('Model Metrikleri Karşılaştırması – Test Seti', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("model_comparison.png kaydedildi.")


In [ ]:
print("\n── Sınıflandırma Raporu (NumPy NN – Seçilen Model) ──")
print(classification_report(y_test, numpy_preds, target_names=['Sahte', 'Gerçek']))


## 8. Sonuç Özeti

In [ ]:
print("\n" + "=" * 70)
print("  YZM304 Derin Öğrenme – I. Proje Ödevi – Sonuç Özeti")
print("=" * 70)
print(f"  Veri Seti       : BankNote Authentication (1372 örnek, 4 özellik)")
print(f"  Lab Hiperparams : seed={SEED}, init=*0.01, lr={LR}, n_steps={N_STEPS}")
print(f"  Seçilen Model   : {best_name}  |  Mimari: {best_model.layer_sizes}")
print(f"  Kayıp Fonks.    : Binary Cross-Entropy (lab ile aynı)")
print(f"  Optimizer       : SGD (lab ile aynı) + L2 Reg. + Mini-Batch (PyTorch)")
print("─" * 70)
print(f"{'Model':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("─" * 70)
for name, row in results_df.iterrows():
    print(f"{name:<15} {row.accuracy:>10.4f} {row.precision:>10.4f} "
          f"{row.recall:>10.4f} {row.f1:>10.4f}")
print("=" * 70)
